# RAG

In [2]:
import os

from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types

import chromadb
import numpy as np

ModuleNotFoundError: No module named 'dotenv'

In [182]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GEMINI_API_KEY")
)

print("Gemini client created")

Gemini client created


## Read the documents

In [183]:
data_dir = Path("data")

documents = []

for file in data_dir.glob("*.txt"):
    text = file.read_text(encoding="utf-8")

    documents.append({
        "source": file.name,
        "text": text
    })

print("Loaded documents:", len(documents))

for document in documents:
    print(document["source"])

Loaded documents: 3
machine_learning.txt
python.txt
rag.txt


## Split the documents into chunks

In [184]:
def chunk_text(text, chunk_size=300, chunk_overlap=50):
    chunks = []

    start = 0

    while start < len(text):
        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - chunk_overlap

    return chunks

In [185]:
chunks = []

for document in documents:
    document_chunks = chunk_text(
        document["text"],
        chunk_size=300,
        chunk_overlap=50
    )

    for chunk_id, chunk in enumerate(document_chunks):
        chunks.append({
            "source": document["source"],
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Created chunks:", len(chunks))

Created chunks: 5


In [186]:
for chunk in chunks:
    print("\n--------------------")
    print("Source:", chunk["source"])
    print("Chunk ID:", chunk["chunk_id"])
    print(chunk["text"])


--------------------
Source: machine_learning.txt
Chunk ID: 0
Machine learning is a branch of artificial intelligence.

Machine learning systems learn patterns from data and use those
patterns to make predictions or decisions.

Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.

--------------------
Source: machine_learning.txt
Chunk ID: 1
 and reinforcement learning.

--------------------
Source: python.txt
Chunk ID: 0
Python is a high-level programming language.

Python is commonly used for web development, data science,
machine learning, automation, scripting, and artificial intelligence.

Python uses indentation to define blocks of code.

--------------------
Source: rag.txt
Chunk ID: 0
RAG stands for Retrieval-Augmented Generation.

RAG combines information retrieval with a language model.
Instead of asking the language model to answer only from its
training data, the system first retrieves relevant information
from an

## Create embeddings

In [187]:
def get_embedding(text):
    response = client.models.embed_content(
        model="gemini-embedding-2",
        contents=text,
        config=types.EmbedContentConfig(
            output_dimensionality=768
        )
    )

    return response.embeddings[0].values

In [188]:
for chunk in chunks:
    chunk["embedding"] = get_embedding(chunk["text"])

print("Embeddings created")

Embeddings created


In [189]:
for chunk in chunks:
    print(
        "Source:", chunk["source"],
        "| Chunk:", chunk["chunk_id"],
        "| Embedding size:", len(chunk["embedding"])
    )

Source: machine_learning.txt | Chunk: 0 | Embedding size: 768
Source: machine_learning.txt | Chunk: 1 | Embedding size: 768
Source: python.txt | Chunk: 0 | Embedding size: 768
Source: rag.txt | Chunk: 0 | Embedding size: 768
Source: rag.txt | Chunk: 1 | Embedding size: 768


## ChromaDB 

In [190]:
chroma_client = chromadb.PersistentClient(
    path="./chroma_db"
)

collection = chroma_client.get_or_create_collection(
    name="my_rag"
)

In [191]:
collection.add(
    ids=[str(i) for i in range(len(chunks))],

    embeddings=[
        chunk["embedding"]
        for chunk in chunks
    ],

    documents=[
        chunk["text"]
        for chunk in chunks
    ],

    metadatas=[
        {
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"]
        }
        for chunk in chunks
    ]
)

User Query

In [192]:
question = "What is RAG?"

question_embedding = get_embedding(question)

question_vector = np.array(
    [question_embedding],
    dtype="float32"
)

In [193]:
results = collection.query(
    query_embeddings=[question_embedding],
    n_results=10
)

retrieved_chunks = results["documents"][0]
retrieved_metadata = results["metadatas"][0]

In [194]:
for chunk, metadata in zip(
    retrieved_chunks,
    retrieved_metadata
):
    print("\n--------------------")
    print("Source:", metadata["source"])
    print("Chunk:", metadata["chunk_id"])
    print(chunk)


--------------------
Source: rag.txt
Chunk: 0
RAG stands for Retrieval-Augmented Generation.

RAG combines information retrieval with a language model.
Instead of asking the language model to answer only from its
training data, the system first retrieves relevant information
from an external knowledge base.

The retrieved information is then pr

--------------------
Source: rag.txt
Chunk: 1
wledge base.

The retrieved information is then provided to the language model
as context.

A basic RAG pipeline contains document loading, chunking,
embedding generation, vector storage, retrieval, and generation.

--------------------
Source: machine_learning.txt
Chunk: 1
 and reinforcement learning.

--------------------
Source: machine_learning.txt
Chunk: 0
Machine learning is a branch of artificial intelligence.

Machine learning systems learn patterns from data and use those
patterns to make predictions or decisions.

Common types of machine learning include supervised learning,
unsupervised 

## Build the context with citation

In [195]:
context_parts = []

for chunk, metadata in zip(
    retrieved_chunks,
    retrieved_metadata
):
    context_parts.append(
        f"""
[Source: {metadata["source"]}, Chunk: {metadata["chunk_id"]}]

{chunk}
"""
    )

context = "\n\n".join(context_parts)

print(context)


[Source: rag.txt, Chunk: 0]

RAG stands for Retrieval-Augmented Generation.

RAG combines information retrieval with a language model.
Instead of asking the language model to answer only from its
training data, the system first retrieves relevant information
from an external knowledge base.

The retrieved information is then pr



[Source: rag.txt, Chunk: 1]

wledge base.

The retrieved information is then provided to the language model
as context.

A basic RAG pipeline contains document loading, chunking,
embedding generation, vector storage, retrieval, and generation.



[Source: machine_learning.txt, Chunk: 1]

 and reinforcement learning.



[Source: machine_learning.txt, Chunk: 0]

Machine learning is a branch of artificial intelligence.

Machine learning systems learn patterns from data and use those
patterns to make predictions or decisions.

Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.



[Source: python.txt, 

## Build the prompt

In [196]:
prompt = f"""
You are a helpful assistant.

Answer the question using only the provided context.

For every important claim, include the source in this format:
[Source: filename, Chunk: number]

If the answer cannot be found in the context,
say "I don't know based on the provided documents."

Context:
{context}

Question:
{question}
"""

## Send context to Gemini

In [197]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=prompt
)

print(response.text)

RAG stands for Retrieval-Augmented Generation [Source: rag.txt, Chunk: 0]. It is a system that combines information retrieval with a language model, where relevant information is retrieved from an external knowledge base and provided to the language model as context instead of relying solely on the model's training data [Source: rag.txt, Chunk: 0] [Source: rag.txt, Chunk: 1]. A basic RAG pipeline consists of document loading, chunking, embedding generation, vector storage, retrieval, and generation [Source: rag.txt, Chunk: 1].


## Reranking

In [198]:
def rerank_chunks(question, chunks, metadata):

    scored_chunks = []

    for chunk, meta in zip(chunks, metadata):

        prompt = f"""
You are a relevance scorer.

Question:
{question}

Document chunk:
{chunk}

Give a relevance score from 0 to 10.

10 = directly answers the question
7-9 = highly relevant
4-6 = somewhat relevant
1-3 = barely relevant
0 = unrelated

Return ONLY the number.
"""

        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=prompt
        )

        try:
            score = float(response.text.strip())
        except ValueError:
            score = 0

        scored_chunks.append({
            "text": chunk,
            "source": meta["source"],
            "chunk_id": meta["chunk_id"],
            "score": score
        })

    scored_chunks.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return scored_chunks

In [199]:
ranked_chunks = rerank_chunks(
    question,
    retrieved_chunks,
    retrieved_metadata
)

In [200]:
for chunk in ranked_chunks:
    print(
        "Score:", chunk["score"],
        "| Source:", chunk["source"],
        "| Chunk:", chunk["chunk_id"]
    )

Score: 9.0 | Source: rag.txt | Chunk: 0
Score: 9.0 | Source: rag.txt | Chunk: 1
Score: 0.0 | Source: machine_learning.txt | Chunk: 1
Score: 0.0 | Source: machine_learning.txt | Chunk: 0
Score: 0.0 | Source: python.txt | Chunk: 0


In [201]:
for chunk in ranked_chunks:
    print("\n----------------")
    print("Score:", chunk["score"])
    print("Source:", chunk["source"])
    print("Chunk:", chunk["chunk_id"])
    print(chunk["text"])


----------------
Score: 9.0
Source: rag.txt
Chunk: 0
RAG stands for Retrieval-Augmented Generation.

RAG combines information retrieval with a language model.
Instead of asking the language model to answer only from its
training data, the system first retrieves relevant information
from an external knowledge base.

The retrieved information is then pr

----------------
Score: 9.0
Source: rag.txt
Chunk: 1
wledge base.

The retrieved information is then provided to the language model
as context.

A basic RAG pipeline contains document loading, chunking,
embedding generation, vector storage, retrieval, and generation.

----------------
Score: 0.0
Source: machine_learning.txt
Chunk: 1
 and reinforcement learning.

----------------
Score: 0.0
Source: machine_learning.txt
Chunk: 0
Machine learning is a branch of artificial intelligence.

Machine learning systems learn patterns from data and use those
patterns to make predictions or decisions.

Common types of machine learning include superv

## Top 3

In [202]:
top_chunks = ranked_chunks[:3]

In [203]:
context_parts = []

for chunk in top_chunks:

    context_parts.append(
        f"""
[Source: {chunk["source"]}, Chunk: {chunk["chunk_id"]}]

{chunk["text"]}
"""
    )

context = "\n\n".join(context_parts)

print(context)


[Source: rag.txt, Chunk: 0]

RAG stands for Retrieval-Augmented Generation.

RAG combines information retrieval with a language model.
Instead of asking the language model to answer only from its
training data, the system first retrieves relevant information
from an external knowledge base.

The retrieved information is then pr



[Source: rag.txt, Chunk: 1]

wledge base.

The retrieved information is then provided to the language model
as context.

A basic RAG pipeline contains document loading, chunking,
embedding generation, vector storage, retrieval, and generation.



[Source: machine_learning.txt, Chunk: 1]

 and reinforcement learning.



# RAG Evaluation

## Create test questions

In [204]:
test_questions = [
    {
        "question": "What does RAG stand for?",
        "expected_source": "rag.txt"
    },
    {
        "question": "What is Python commonly used for?",
        "expected_source": "python.txt"
    },
    {
        "question": "What are the common types of machine learning?",
        "expected_source": "machine_learning.txt"
    }
]

## Test retrieval

In [205]:
for test in test_questions:

    question = test["question"]

    question_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=3
    )

    retrieved_metadata = results["metadatas"][0]

    print("\nQuestion:", question)

    for metadata in retrieved_metadata:
        print(
            "Source:",
            metadata["source"],
            "| Chunk:",
            metadata["chunk_id"]
        )


Question: What does RAG stand for?
Source: rag.txt | Chunk: 0
Source: rag.txt | Chunk: 1
Source: machine_learning.txt | Chunk: 1

Question: What is Python commonly used for?
Source: python.txt | Chunk: 0
Source: machine_learning.txt | Chunk: 0
Source: machine_learning.txt | Chunk: 1

Question: What are the common types of machine learning?
Source: machine_learning.txt | Chunk: 0
Source: machine_learning.txt | Chunk: 1
Source: python.txt | Chunk: 0


## Measure retrieval accuracy

In [206]:
for test in test_questions:

    question = test["question"]
    expected_source = test["expected_source"]

    question_embedding = get_embedding(question)

    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=3
    )

    retrieved_metadata = results["metadatas"][0]

    sources = [
        metadata["source"]
        for metadata in retrieved_metadata
    ]

    if expected_source in sources:
        print("✅ PASS:", question)
    else:
        print("❌ FAIL:", question)

✅ PASS: What does RAG stand for?
✅ PASS: What is Python commonly used for?
✅ PASS: What are the common types of machine learning?


# Hybrid Search

## Create keyword search

In [211]:
import re

def keyword_search(query, chunks, top_k=5):

    stop_words = {
        "what", "is", "the", "a", "an",
        "of", "to", "in", "and", "for",
        "on", "with"
    }

    query_words = re.findall(
        r"\b\w+\b",
        query.lower()
    )

    query_words = [
        word
        for word in query_words
        if word not in stop_words
    ]

    scored_chunks = []

    for chunk in chunks:

        text_words = re.findall(
            r"\b\w+\b",
            chunk["text"].lower()
        )

        score = 0

        for word in query_words:
            if word in text_words:
                score += 1

        scored_chunks.append({
            "chunk": chunk,
            "score": score
        })

    scored_chunks.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return scored_chunks[:top_k]

In [212]:
results = keyword_search(
    "What is RAG?",
    chunks,
    top_k=3
)

for result in results:

    print(
        "Score:",
        result["score"],
        "| Source:",
        result["chunk"]["source"]
    )

    print(result["chunk"]["text"])
    print()

Score: 1 | Source: rag.txt
RAG stands for Retrieval-Augmented Generation.

RAG combines information retrieval with a language model.
Instead of asking the language model to answer only from its
training data, the system first retrieves relevant information
from an external knowledge base.

The retrieved information is then pr

Score: 1 | Source: rag.txt
wledge base.

The retrieved information is then provided to the language model
as context.

A basic RAG pipeline contains document loading, chunking,
embedding generation, vector storage, retrieval, and generation.

Score: 0 | Source: machine_learning.txt
Machine learning is a branch of artificial intelligence.

Machine learning systems learn patterns from data and use those
patterns to make predictions or decisions.

Common types of machine learning include supervised learning,
unsupervised learning, and reinforcement learning.

